# Alternating parameter recovery

Loads every JSONL result written by `scripts/5_alternating_parameter_recovery.sh`. The cells intentionally stop at tidy tables so plots and analyses can be added here.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path.cwd().parent / 'derived' / 'results' / 'alternating_parameter_recovery'
RESULTS_DIR

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

paths = sorted(RESULTS_DIR.glob('*.jsonl'))
if not paths:
    raise FileNotFoundError(f'No JSONL result files found in {RESULTS_DIR}')

records = [
    {'result_file': path.name, 'result_path': path, **row}
    for path in paths
    for row in load_jsonl(path)
]

print(f'Loaded {len(records):,} rows from {len(paths)} result files.')

In [ ]:
# One row per training epoch. `parameter_trace` contains the cheap recovery
# diagnostics recorded even when full validation was not due.
epoch_table = pd.json_normalize(records, sep='_').sort_values(
    ['result_file', 'epoch'], kind='stable'
).reset_index(drop=True)

trace_columns = [
    'result_file', 'run', 'seed', 'epoch', 'alternating_phase',
    *[column for column in epoch_table if column.startswith('parameter_trace_')],
]
parameter_traces = epoch_table.loc[:, trace_columns].copy()

parameter_traces.head()

In [ ]:
# Validation/test columns are present only at evaluation epochs.
evaluation_columns = [
    'result_file', 'run', 'seed', 'epoch', 'alternating_phase',
    *[column for column in epoch_table if column.startswith(('val_', 'test_', 'rollout_val_', 'rollout_test_'))],
]
evaluations = epoch_table.loc[epoch_table.get('val_loss').notna(), evaluation_columns].copy()

print(f'Found {len(evaluations):,} evaluated epochs.')
evaluations.head()

In [ ]:
# Convenient file-level metadata for grouping custom analyses.
runs = (
    epoch_table.groupby('result_file', as_index=False)
    .agg(
        seed=('seed', 'first'),
        epochs_logged=('epoch', 'max'),
        phases=('alternating_phase', lambda values: tuple(pd.unique(values.dropna()))),
    )
)
runs